# Ingest — commercial terms

Writes a one-row `commercial_terms` table that the template reads **instead of** its parameters.

**This notebook is optional and unlike the other four it ingests no consumption data.** It exists
for one reason: the rate you actually pay is not the rate Microsoft publishes. List price is
$0.01 per credit. An enterprise agreement may not be. Every cost figure in Consumption Central is
`credits x rate`, so a wrong rate is wrong everywhere, quietly, by a constant factor.

Azure Cost Management knows the real number, because it reports what you were charged and how
much you consumed. Divide one by the other.

| Column written | Meaning |
| --- | --- |
| `credit_rate` | Pay-as-you-go cost per credit, derived from actual spend |
| `prepaid_credit_rate` | Effective cost per pre-purchased credit — manual, see below |
| `prepaid_credit_balance` | Credits bought up front — manual |
| `github_business_seat` | Monthly price per Copilot Business seat — manual |
| `github_enterprise_seat` | Monthly price per Copilot Enterprise seat — manual |

Any column you omit falls back to the template parameter. A table containing only `credit_rate` is
a perfectly reasonable output of this notebook.

## What is and is not automatable

**Derivable from Azure:** the pay-as-you-go credit rate. Copilot credits — including Cowork and
Work IQ, which bill under the Copilot Studio meter — appear on the Azure invoice, so Cost
Management can be asked for cost and quantity on that meter.

**Not derivable:** everything GitHub. GitHub Copilot seats are billed by GitHub, not Azure, and
GitHub's billing API reports credit consumption, not what you pay per seat. Those stay manual.
Prepaid capacity likewise — a capacity pack is a purchase, not a meter, and the effective rate
depends on how much of it you use.

So this closes the gap that matters most and leaves the rest honest.

## Before you run it

You need **Cost Management Reader** on the billing scope. That is a real permission grant and
often the reason this notebook does not get used; if it is refused, delete this notebook and set
`CreditRate` in the template by hand. Nothing else breaks.

> **A caution about what this number means.** Cost / quantity is a *blended* rate over whatever
> window you query. If your rate changed mid-period, or you consumed under two agreements, the
> answer is an average of both and is not the rate for any single credit. The notebook prints the
> window and the row count it used so you can see what went into it. Query one clean month.


## Parameters

`SCOPE` is the awkward one. It is an ARM resource path, and which one depends on how you buy
Azure. In order of preference:

```
Enterprise Agreement   /providers/Microsoft.Billing/billingAccounts/{billingAccountId}
MCA billing profile    /providers/Microsoft.Billing/billingAccounts/{id}/billingProfiles/{profileId}
Single subscription    /subscriptions/{subscriptionId}
```

Find yours in the Azure portal under **Cost Management + Billing**; the account or profile ID is
in the URL. A subscription scope only sees that subscription's share, which is fine if Copilot
credits are billed there and wrong if they are pooled elsewhere.


In [ ]:
# --- parameters -----------------------------------------------------------
LAKEHOUSE_TABLE = "commercial_terms"

# Cost Management scope. See the note above - this is the part people get wrong.
SCOPE = "/subscriptions/00000000-0000-0000-0000-000000000000"

# The window to derive the rate from. One complete, recent month is the right
# answer: long enough to be a real average, short enough not to straddle a
# price change.
FROM_DATE = "2026-07-01"
TO_DATE   = "2026-07-31"

# ActualCost is what you were invoiced. Switch to "AmortizedCost" if you hold
# prepaid capacity: usage covered by a reservation shows as $0 under ActualCost,
# which would derive a rate of zero and make every cost in the report vanish.
COST_TYPE = "ActualCost"

# The meter. Cowork, Work IQ and Copilot Studio all bill through this one -
# there is no separate Cowork meter, which surprises people.
SERVICE_NAME = "Microsoft Copilot Studio"
METER_FILTER = "Copilot Credit"      # substring match on meterName

# Values Azure cannot tell us. Leave a value as None to omit the column
# entirely, which makes the template fall back to its parameter.
PREPAID_CREDIT_RATE    = None    # e.g. 0.008
PREPAID_CREDIT_BALANCE = None    # e.g. 500000
GITHUB_BUSINESS_SEAT   = 19.0
GITHUB_ENTERPRISE_SEAT = 39.0

# Safety net. If the derived rate falls outside this range something is wrong -
# the wrong meter, the wrong scope, a partial month - and a silently wrong rate
# is worse than no rate. The notebook refuses rather than writing it.
RATE_SANITY_RANGE = (0.001, 0.05)

## Authenticate

Inside Fabric, `notebookutils.credentials.getToken` gives a token for the workspace identity —
no secret to store. That identity still needs Cost Management Reader granted to it explicitly.

Outside Fabric, fall back to `azure-identity`.


In [ ]:
import json
import requests

ARM = "https://management.azure.com"


def get_token():
    """Workspace identity inside Fabric, DefaultAzureCredential outside it."""
    try:
        import notebookutils
        return notebookutils.credentials.getToken(ARM)
    except Exception:
        from azure.identity import DefaultAzureCredential
        return DefaultAzureCredential().get_token(
            ARM + "/.default").token


TOKEN = get_token()
print("token acquired:", bool(TOKEN))

## Query cost and quantity

One POST to the Cost Management query API, grouped by meter, filtered to the Copilot service.

`UsageQuantity` is the credit count and `Cost` is what you were charged for it. Both are needed;
either alone tells you nothing about the rate.


In [ ]:
url = (f"{ARM}{SCOPE}/providers/Microsoft.CostManagement/query"
       "?api-version=2023-11-01")

body = {
    "type": COST_TYPE,
    "timeframe": "Custom",
    "timePeriod": {"from": FROM_DATE, "to": TO_DATE},
    "dataset": {
        "granularity": "None",
        "aggregation": {
            "totalCost":     {"name": "Cost",          "function": "Sum"},
            "totalQuantity": {"name": "UsageQuantity", "function": "Sum"},
        },
        "grouping": [
            {"type": "Dimension", "name": "MeterName"},
        ],
        "filter": {
            "dimensions": {
                "name": "ServiceName",
                "operator": "In",
                "values": [SERVICE_NAME],
            }
        },
    },
}

resp = requests.post(
    url,
    headers={"Authorization": f"Bearer {TOKEN}",
             "Content-Type": "application/json"},
    data=json.dumps(body),
    timeout=120,
)

if resp.status_code == 403:
    raise SystemExit(
        "403 from Cost Management. The identity running this notebook does not "
        "have Cost Management Reader on " + SCOPE + ". Grant it, or set "
        "CreditRate in the template by hand and skip this notebook."
    )
resp.raise_for_status()

payload = resp.json()
cols = [c["name"] for c in payload["properties"]["columns"]]
rows = payload["properties"]["rows"]
print(f"{len(rows)} meter rows returned")
for r in rows:
    print(dict(zip(cols, r)))

## Derive the rate

Keep only meters whose name contains the credit filter, sum both figures across them, divide.

Summing across meters before dividing — rather than averaging per-meter rates — is deliberate:
it weights by consumption, which is what a blended rate means.


In [ ]:
def col(name):
    """Column order in the response is not guaranteed."""
    for i, c in enumerate(cols):
        if c.lower() == name.lower():
            return i
    raise KeyError(f"{name} not in response columns {cols}")


i_meter = col("MeterName")
i_cost  = col("Cost")
i_qty   = col("UsageQuantity")

matched = [r for r in rows
           if METER_FILTER.lower() in str(r[i_meter]).lower()]

if not matched:
    raise SystemExit(
        f"No meter matching '{METER_FILTER}' under service '{SERVICE_NAME}' "
        f"between {FROM_DATE} and {TO_DATE}.\n"
        "Either there was no pay-as-you-go credit consumption in that window, "
        "or the scope is wrong. The meters that DID come back are printed "
        "above - if one of them looks like the right one, put a distinctive "
        "piece of its name in METER_FILTER."
    )

total_cost = sum(float(r[i_cost]) for r in matched)
total_qty  = sum(float(r[i_qty])  for r in matched)

print("meters used:", [r[i_meter] for r in matched])
print(f"cost     {total_cost:,.2f}")
print(f"quantity {total_qty:,.0f}")

if total_qty <= 0:
    raise SystemExit(
        "Quantity is zero, so no rate can be derived. Credits were billed at "
        "no quantity, or the window contains no consumption."
    )

credit_rate = total_cost / total_qty
print(f"\nderived rate: {credit_rate:.6f} per credit")
print(f"list price:   0.010000 per credit")
print(f"difference:   {(credit_rate / 0.01 - 1) * 100:+.1f}%")

lo, hi = RATE_SANITY_RANGE
if not (lo <= credit_rate <= hi):
    raise SystemExit(
        f"Derived rate {credit_rate:.6f} is outside the sanity range "
        f"{lo}-{hi} and will NOT be written. A rate this far from list price "
        "is much more likely to be a wrong meter or a partial month than a "
        "real commercial term. Check the meters listed above."
    )

## Write the table

Overwrite, not merge. This is a settings table with one row — there is no history to preserve, and
appending would leave the template reading whichever row happened to come first.


In [ ]:
from datetime import datetime, timezone
from pyspark.sql import Row
from pyspark.sql.types import (StructType, StructField, DoubleType,
                               TimestampType)

values = {
    "credit_rate": float(credit_rate),
    "prepaid_credit_rate": PREPAID_CREDIT_RATE,
    "prepaid_credit_balance": PREPAID_CREDIT_BALANCE,
    "github_business_seat": GITHUB_BUSINESS_SEAT,
    "github_enterprise_seat": GITHUB_ENTERPRISE_SEAT,
}

# A None means "we do not know" - omit the column so the template falls back to
# its parameter. Writing null would be read as a value and override with blank.
present = {k: v for k, v in values.items() if v is not None}

schema = StructType(
    [StructField(k, DoubleType(), True) for k in present]
    + [StructField("_loaded_at", TimestampType(), False)]
)

row = Row(**present, _loaded_at=datetime.now(timezone.utc))
df = spark.createDataFrame([row], schema=schema)

df.write.format("delta").mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(LAKEHOUSE_TABLE)

print(f"wrote {LAKEHOUSE_TABLE}:")
df.show(truncate=False)
print("omitted (template parameter will be used):",
      sorted(set(values) - set(present)) or "none")

## Check it landed

The template reads row zero. If this shows one row with the columns you expect, the report will
pick them up on its next refresh — no change to the `.pbit`, no parameter to retype.


In [ ]:
spark.sql(f"SELECT * FROM {LAKEHOUSE_TABLE}").show(truncate=False)

---

## If this notebook does not work for you

That is a supported outcome. The template's parameters are the fallback and they are not a
second-class path — they are what the CSV template uses exclusively.

Delete the `commercial_terms` table and Consumption Central goes back to reading `CreditRate` and the seat
prices from the template. Nothing else changes.

Related reading: [`docs/COMMERCIAL-TERMS.md`](../../docs/COMMERCIAL-TERMS.md) covers where every
commercial figure comes from and which ones any API can answer.
